In [1]:
import pandas as pd
import numpy as np
import os
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import joblib

print("Engineering Live Ball-by-Ball Data and Training Phase Models...")

# 1. Load Data
matches = pd.read_csv('../data/raw/matches.csv')
deliveries = pd.read_csv('../data/raw/deliveries.csv')

# Clean base matches data
matches.dropna(subset=['winner'], inplace=True)

# 2. Get 1st Innings Target Score
inning1 = deliveries[deliveries['inning'] == 1].groupby('match_id')['total_runs'].sum().reset_index()
inning1.rename(columns={'total_runs': 'target'}, inplace=True)
inning1['target'] = inning1['target'] + 1 # Target is 1st innings runs + 1

# 3. Filter to 2nd Innings (The Chase)
chase_df = deliveries[deliveries['inning'] == 2].copy()
chase_df = chase_df.merge(inning1, on='match_id')
chase_df = chase_df.merge(matches[['id', 'winner', 'team2']], left_on='match_id', right_on='id')

# 4. Calculate Ball-by-Ball Match State Features
chase_df['runs_scored'] = chase_df.groupby('match_id')['total_runs'].cumsum()

# Handle wickets
if 'is_wicket' in chase_df.columns:
    chase_df['wickets_lost'] = chase_df.groupby('match_id')['is_wicket'].cumsum()
else:
    chase_df['is_wicket'] = chase_df['player_dismissed'].notna().astype(int)
    chase_df['wickets_lost'] = chase_df.groupby('match_id')['is_wicket'].cumsum()

chase_df['balls_bowled'] = chase_df.groupby('match_id').cumcount() + 1

# The Core ML Features
chase_df['runs_required'] = chase_df['target'] - chase_df['runs_scored']
chase_df['runs_required'] = chase_df['runs_required'].apply(lambda x: x if x > 0 else 0)
chase_df['balls_left'] = 120 - chase_df['balls_bowled']
chase_df['balls_left'] = chase_df['balls_left'].apply(lambda x: x if x > 0 else 1) # Prevent division by zero
chase_df['wickets_in_hand'] = 10 - chase_df['wickets_lost']

# Rates
chase_df['crr'] = (chase_df['runs_scored'] / chase_df['balls_bowled']) * 6
chase_df['rrr'] = (chase_df['runs_required'] / chase_df['balls_left']) * 6

# The Target Variable (Did the chasing team win?)
chase_df['chase_successful'] = (chase_df['batting_team'] == chase_df['winner']).astype(int)

# 5. Define the Phases
def get_phase(over):
    if over <= 5: return 'Powerplay'
    elif over <= 14: return 'Middle'
    else: return 'Death'

chase_df['phase'] = chase_df['over'].apply(get_phase)

# 6. Train a Model for Each Phase
phases = ['Powerplay', 'Middle', 'Death']
features = ['runs_required', 'balls_left', 'wickets_in_hand', 'crr', 'rrr']

os.makedirs('../models', exist_ok=True)

for phase in phases:
    print(f"\n--- Training {phase} Model ---")
    
    # Isolate data for this specific phase
    phase_data = chase_df[chase_df['phase'] == phase].copy()
    
    X = phase_data[features]
    y = phase_data['chase_successful']
    
    # We use random split here because ball-by-ball data is massive and highly repetitive chronologically
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Initialize XGBoost for the Phase
    model = xgb.XGBClassifier(n_estimators=150, learning_rate=0.05, max_depth=6, eval_metric='logloss', random_state=42)
    model.fit(X_train, y_train)
    
    # Evaluate
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"✅ {phase} Model Accuracy: {acc * 100:.2f}%")
    
    # Save the specialized model
    joblib.dump(model, f'../models/live_chase_{phase.lower()}.pkl')

joblib.dump(features, '../models/live_chase_features.pkl')
print("\n🎉 All 3 phase-specific Live Models successfully trained and saved!")

Engineering Live Ball-by-Ball Data and Training Phase Models...

--- Training Powerplay Model ---
✅ Powerplay Model Accuracy: 72.99%

--- Training Middle Model ---
✅ Middle Model Accuracy: 79.86%

--- Training Death Model ---
✅ Death Model Accuracy: 88.05%

🎉 All 3 phase-specific Live Models successfully trained and saved!
